# Crop Disease Detection — EfficientNetB3

**Model:** EfficientNetB3 (Transfer Learning from ImageNet)  
**Dataset:** New Plant Diseases Dataset (PlantVillage) — 87,000 images, 38 classes  
**Platform:** Google Colab (T4 GPU)  

The idea is simple — EfficientNetB3 is already trained on millions of images so it knows how to "see". We just teach it the specific task of identifying plant diseases by adding our own classification layers on top.

## Step 1: Check GPU + Download Dataset

In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# If GPU list is empty — go to Runtime > Change runtime type > T4 GPU

In [ ]:
from google.colab import files
import os

# Upload kaggle.json to download the dataset
# Get it from: kaggle.com > Account > Settings > Create New Token
files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle setup done!')

In [ ]:
# Download dataset (~2.7GB, takes 2-3 minutes)
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset
!unzip -q new-plant-diseases-dataset.zip
print('Dataset ready!')

In [ ]:
import os

TRAIN_DIR = 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'
VALID_DIR = 'New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid'

classes = os.listdir(TRAIN_DIR)
print(f'Total classes: {len(classes)}')
print(f'Classes: {sorted(classes)}')

## Step 2: Load and Augment the Data

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = (300, 300)  # EfficientNetB3 native input size
BATCH_SIZE = 32

# Training data — we apply random augmentations so the model
# sees slightly different versions of images every epoch
# This helps it generalize better to unseen images
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # scales pixels to [-1, 1] as EfficientNet expects
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.25,
    horizontal_flip=True,
    vertical_flip=False,         # leaves don't grow upside down
    brightness_range=[0.8, 1.2], # simulate different lighting
    fill_mode='nearest'
)

# Validation data — no augmentation, just preprocessing
# We want to evaluate on clean unmodified images
valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
print(f'Training samples   : {train_generator.samples}')
print(f'Validation samples : {validation_generator.samples}')
print(f'Number of classes  : {NUM_CLASSES}')

## Step 3: Build the Model

In [ ]:
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Load EfficientNetB3 pretrained on ImageNet
# include_top=False removes its original 1000-class classifier
# we'll attach our own 38-class classifier instead
base_model = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_shape=(300, 300, 3)
)

# Freeze the base model — we don't want to change its weights yet
# just train our new layers first
base_model.trainable = False

# Add our custom classification head on top
x = base_model.output
x = GlobalAveragePooling2D()(x)   # flatten the feature map into a vector
x = BatchNormalization()(x)        # stabilize training
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)                # randomly turn off 40% neurons to prevent overfitting
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)  # 38 output probabilities

model = Model(inputs=base_model.input, outputs=output)

print(f'Total params     : {model.count_params():,}')
print(f'Trainable params : {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

## Step 4: Train the Model

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    # stop training if val_accuracy doesn't improve for 5 epochs
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # reduce learning rate if val_loss stops improving for 3 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    # save the best model automatically whenever val_accuracy improves
    ModelCheckpoint(
        'plant_disease_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Starting training...\n')

history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

print(f'\nTraining complete!')
print(f'Best val accuracy: {max(history.history["val_accuracy"]):.4f}')

## Step 5: Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

acc      = history.history['accuracy']
val_acc  = history.history['val_accuracy']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc, 'b-', label='Train Accuracy', linewidth=2)
ax1.plot(epochs, val_acc, 'r-', label='Val Accuracy', linewidth=2)
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

ax2.plot(epochs, loss, 'b-', label='Train Loss', linewidth=2)
ax2.plot(epochs, val_loss, 'r-', label='Val Loss', linewidth=2)
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## Step 6: Evaluate the Model

In [ ]:
print('Final evaluation on validation set:')
loss, accuracy = model.evaluate(validation_generator, verbose=1)
print(f'\nVal Loss     : {loss:.4f}')
print(f'Val Accuracy : {accuracy*100:.2f}%')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Run predictions on all validation images
print('Generating predictions...')
validation_generator.reset()
y_pred = model.predict(validation_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)  # pick highest probability class
y_true = validation_generator.classes

class_names = list(train_generator.class_indices.keys())

report = classification_report(y_true, y_pred_classes, target_names=class_names)
print(report)

with open('classification_report.txt', 'w') as f:
    f.write(report)
print('Saved classification_report.txt')

## Step 7: Save and Download Everything

In [ ]:
import json
from google.colab import files

# Save class indices — maps class number to disease name
# e.g. {"Tomato___Early_blight": 29, ...}
# needed in the app to convert model output number back to disease name
with open('class_indices.json', 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)

# Save model info
metadata = {
    'architecture': 'EfficientNetB3',
    'input_size': [300, 300, 3],
    'num_classes': NUM_CLASSES,
    'preprocessing': 'efficientnet.preprocess_input',
    'val_accuracy': float(max(history.history['val_accuracy']))
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Downloading files...')
files.download('plant_disease_model.keras')
files.download('class_indices.json')
files.download('model_metadata.json')
files.download('training_curves.png')
files.download('classification_report.txt')
print('Done!')